In [ ]:
# ============================================================
# 20 — COMPLEMENTO DO MODELO FINAL
# SHAP + RANKINGS MUNICIPAIS + RESUMO FINAL PARA O TCC
# ============================================================
#
# ESTE CÓDIGO:
#
# NÃO:
#   - treina novamente o Random Forest;
#   - redefine hiperparâmetros;
#   - recalcula previsões de 2024;
#   - recalcula previsões de 2025;
#   - altera o threshold.
#
# ELE:
#   1. localiza a execução final já existente;
#   2. carrega RF_final.joblib;
#   3. carrega amostra_SHAP_2024.parquet;
#   4. corrige sparse -> ndarray float32;
#   5. calcula SHAP;
#   6. gera arquivos 08 a 13;
#   7. reconstrói os rankings municipais;
#   8. acrescenta nomes IBGE quando possível;
#   9. gera arquivos 14 a 20;
#  10. usa métricas 02 a 07 já existentes;
#  11. gera resumo 21 e manifesto 22.
#
# ============================================================


# ============================================================
# 0. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

import os
import glob
import gc
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from scipy import sparse

from IPython.display import display

warnings.filterwarnings("ignore")


# ============================================================
# 2. INSTALAR / IMPORTAR SHAP
# ============================================================

try:

    import shap

except ImportError:

    !pip install -q shap

    import shap


# ============================================================
# 3. CONFIGURAÇÕES
# ============================================================

SEED = 42

TARGET = "Y_doenca"

ALPHA_MUNICIPIO = 1000.0

SHAP_N = 10_000

BOOTSTRAP_SHAP = 200

MIN_N_RANKING = 500

TOP_N_RANKING = 30

BATCH_SIZE = 500_000


PASTA_TCC = (
    "/content/drive/MyDrive/TCC_2"
)


PASTA_BASE = os.path.join(
    PASTA_TCC,
    "dados",
    "RAIS_BASE_MODELO_FINAL_V3"
)


COL_MUN = (
    "Municipio_estabelecimento_codigo"
)


COL_RISCO = (
    "Risco_historico_municipio"
)


COL_LOGN = (
    "Log_n_historico_municipio"
)


NUMERICAS_FINAL = [
    "Idade",
    "Qtd_horas_contratuais",
    "Tempo_emprego_meses",
    COL_RISCO,
    COL_LOGN,
]


CATEGORICAS_FINAL = [
    "UF",
    "Familia_CBO",
    "Sexo_codigo",
    "Tipo_vinculo_macro",
    "Natureza_macro",
    "Tamanho_estabelecimento_codigo",
    "Indicador_deficiencia_codigo",
]


TODAS_VARIAVEIS_FINAL = (
    NUMERICAS_FINAL
    +
    CATEGORICAS_FINAL
)


# ============================================================
# 4. LOCALIZAR AUTOMATICAMENTE A EXECUÇÃO FINAL
# ============================================================
#
# Procuramos especificamente o manifesto criado antes de 2025.
# Isso reduz o risco de utilizar acidentalmente uma execução
# experimental antiga.
# ============================================================

padrao_manifesto = os.path.join(
    PASTA_TCC,
    "resultados",
    "**",
    "04_modelo_congelado_antes_2025.json"
)


manifestos = glob.glob(
    padrao_manifesto,
    recursive=True
)


if not manifestos:

    raise FileNotFoundError(
        "Não encontrei 04_modelo_congelado_antes_2025.json "
        "em TCC_2/resultados."
    )


# Se houver mais de um, usa o mais recentemente modificado.
ARQUIVO_MANIFESTO = max(
    manifestos,
    key=os.path.getmtime
)


PASTA_RESULTADOS = os.path.dirname(
    ARQUIVO_MANIFESTO
)


PASTA_MODELOS = os.path.join(
    PASTA_RESULTADOS,
    "modelos"
)


PASTA_TABELAS = os.path.join(
    PASTA_RESULTADOS,
    "tabelas"
)


PASTA_PREDICOES = os.path.join(
    PASTA_RESULTADOS,
    "predicoes"
)


PASTA_FIGURAS = os.path.join(
    PASTA_RESULTADOS,
    "figuras"
)


for pasta in [
    PASTA_TABELAS,
    PASTA_PREDICOES,
    PASTA_FIGURAS,
]:

    os.makedirs(
        pasta,
        exist_ok=True
    )


print(
    "\nEXECUÇÃO SELECIONADA:"
)

print(
    PASTA_RESULTADOS
)


print(
    "\nManifesto:"
)

print(
    ARQUIVO_MANIFESTO
)


# ============================================================
# 5. LER MANIFESTO CONGELADO
# ============================================================

with open(
    ARQUIVO_MANIFESTO,
    "r",
    encoding="utf-8"
) as f:

    manifesto = json.load(
        f
    )


print(
    "\nMODELO CONGELADO"
)

print(
    "Modelo:",
    manifesto.get(
        "Modelo"
    )
)

print(
    "Treino:",
    manifesto.get(
        "Treino"
    )
)

print(
    "Validação:",
    manifesto.get(
        "Validacao"
    )
)

print(
    "Threshold:",
    manifesto.get(
        "Threshold"
    )
)

print(
    "Alpha:",
    manifesto.get(
        "Alpha_municipal"
    )
)


THRESHOLD_FINAL = float(
    manifesto[
        "Threshold"
    ]
)


# ============================================================
# 6. LOCALIZAR ARTEFATOS NECESSÁRIOS
# ============================================================

ARQ_MODELO_FINAL = os.path.join(
    PASTA_MODELOS,
    "RF_final.joblib"
)


ARQ_AMOSTRA_SHAP = os.path.join(
    PASTA_PREDICOES,
    "amostra_SHAP_2024.parquet"
)


ARQ_METRICAS_2024 = os.path.join(
    PASTA_TABELAS,
    "02_metricas_validacao_2024.csv"
)


ARQ_MESMA_SENS = os.path.join(
    PASTA_TABELAS,
    "03_mesma_sensibilidade_2024.csv"
)


ARQ_METRICAS_2025 = os.path.join(
    PASTA_TABELAS,
    "04_metricas_2025.csv"
)


ARQ_COMPARACAO = os.path.join(
    PASTA_TABELAS,
    "05_comparacao_2024_2025.csv"
)


ARQ_CALIBRACAO = os.path.join(
    PASTA_TABELAS,
    "06_calibracao_resumo_2025.csv"
)


ARQ_CALIBRACAO_DECIS = os.path.join(
    PASTA_TABELAS,
    "07_calibracao_decis_2025.csv"
)


arquivos_obrigatorios = [

    ARQ_MODELO_FINAL,

    ARQ_AMOSTRA_SHAP,

    ARQ_METRICAS_2024,

    ARQ_MESMA_SENS,

    ARQ_METRICAS_2025,

    ARQ_COMPARACAO,

    ARQ_CALIBRACAO,

    ARQ_CALIBRACAO_DECIS,
]


faltantes = [
    x
    for x in arquivos_obrigatorios
    if not os.path.exists(x)
]


if faltantes:

    print(
        "\nARQUIVOS OBRIGATÓRIOS AUSENTES:"
    )

    for x in faltantes:

        print(
            " -",
            x
        )

    raise FileNotFoundError(
        "Existem artefatos necessários ausentes."
    )


print(
    "\nTodos os artefatos necessários foram encontrados."
)


# ============================================================
# 7. CARREGAR MODELO FINAL
# ============================================================

print(
    "\nCarregando RF final..."
)


pacote = joblib.load(
    ARQ_MODELO_FINAL
)


if isinstance(
    pacote,
    dict
):

    rf_final = pacote[
        "modelo"
    ]

    prep_final = pacote[
        "preprocessador"
    ]

else:

    raise RuntimeError(
        "RF_final.joblib não possui a estrutura esperada."
    )


print(
    "Modelo:",
    type(
        rf_final
    )
)


print(
    "Preprocessador:",
    type(
        prep_final
    )
)


# ============================================================
# 8. CARREGAR AMOSTRA SHAP
# ============================================================

print(
    "\nCarregando amostra SHAP..."
)


amostra_shap = pd.read_parquet(
    ARQ_AMOSTRA_SHAP
)


print(
    "Registros:",
    f"{len(amostra_shap):,}"
)


print(
    "Colunas:",
    len(
        amostra_shap.columns
    )
)


if TARGET not in amostra_shap.columns:

    raise RuntimeError(
        f"{TARGET} não existe na amostra SHAP."
    )


# ============================================================
# 9. TRANSFORMAR AMOSTRA
# ============================================================

print(
    "\nTransformando amostra com o preprocessador congelado..."
)


X_sparse = prep_final.transform(
    amostra_shap
)


print(
    "Tipo original:",
    type(
        X_sparse
    )
)


print(
    "Shape:",
    X_sparse.shape
)


print(
    "Dtype original:",
    getattr(
        X_sparse,
        "dtype",
        None
    )
)


# ============================================================
# 10. CORREÇÃO DEFINITIVA DO ERRO dtype('O')
# ============================================================

if sparse.issparse(
    X_sparse
):

    print(
        "\nMatriz sparse detectada."
    )

    print(
        "Convertendo SOMENTE a amostra SHAP "
        "para ndarray float32..."
    )


    X_shap = (
        X_sparse
        .astype(
            np.float32
        )
        .toarray()
    )


else:

    X_shap = np.asarray(
        X_sparse,
        dtype=np.float32
    )


del X_sparse

gc.collect()


print(
    "\nMATRIZ SHAP"
)

print(
    "Tipo:",
    type(
        X_shap
    )
)

print(
    "Shape:",
    X_shap.shape
)

print(
    "Dtype:",
    X_shap.dtype
)


n_nan = int(
    np.isnan(
        X_shap
    ).sum()
)


n_inf = int(
    np.isinf(
        X_shap
    ).sum()
)


print(
    "NaN:",
    n_nan
)

print(
    "Inf:",
    n_inf
)


if n_nan > 0:

    raise RuntimeError(
        "Existem NaN na matriz entregue ao SHAP."
    )


if n_inf > 0:

    raise RuntimeError(
        "Existem valores infinitos na matriz SHAP."
    )


# ============================================================
# 11. NOMES DAS FEATURES
# ============================================================

feature_names = np.asarray(
    prep_final.get_feature_names_out(),
    dtype=object
)


if len(
    feature_names
) != X_shap.shape[1]:

    raise RuntimeError(
        "Quantidade de nomes diferente do número "
        "de colunas da matriz."
    )


print(
    "\nFeatures transformadas:",
    len(
        feature_names
    )
)


# ============================================================
# 12. SHAP
# ============================================================

ARQ_SHAP_ARRAY = os.path.join(
    PASTA_PREDICOES,
    "SHAP_classe_positiva_2024.npy"
)


if os.path.exists(
    ARQ_SHAP_ARRAY
):

    print(
        "\nValores SHAP já encontrados."
    )

    print(
        "Reutilizando:"
    )

    print(
        ARQ_SHAP_ARRAY
    )


    shap_pos = np.load(
        ARQ_SHAP_ARRAY
    )


else:

    print(
        "\nCalculando TreeSHAP..."
    )


    explainer = shap.TreeExplainer(
        rf_final
    )


    inicio = time.time()


    shap_raw = explainer.shap_values(
        X_shap,
        check_additivity=False
    )


    minutos = (
        time.time()
        -
        inicio
    ) / 60


    print(
        f"SHAP concluído em {minutos:.2f} minutos."
    )


    # --------------------------------------------------------
    # Compatibilidade entre versões do SHAP
    # --------------------------------------------------------

    if isinstance(
        shap_raw,
        list
    ):

        if len(
            shap_raw
        ) < 2:

            raise RuntimeError(
                "Saída SHAP não contém classe positiva."
            )


        shap_pos = np.asarray(
            shap_raw[
                1
            ],
            dtype=np.float32
        )


    else:

        shap_array = np.asarray(
            shap_raw
        )


        print(
            "Shape bruto:",
            shap_array.shape
        )


        if (
            shap_array.ndim == 3
            and
            shap_array.shape[
                2
            ] == 2
        ):

            shap_pos = (
                shap_array[
                    :,
                    :,
                    1
                ]
                .astype(
                    np.float32
                )
            )


        elif shap_array.ndim == 2:

            shap_pos = (
                shap_array
                .astype(
                    np.float32
                )
            )


        else:

            raise RuntimeError(
                f"Formato SHAP inesperado: "
                f"{shap_array.shape}"
            )


    np.save(
        ARQ_SHAP_ARRAY,
        shap_pos
    )


    print(
        "SHAP salvo em:"
    )

    print(
        ARQ_SHAP_ARRAY
    )


if shap_pos.shape != X_shap.shape:

    raise RuntimeError(
        "Shape SHAP diferente da matriz transformada.\n"
        f"SHAP: {shap_pos.shape}\n"
        f"X: {X_shap.shape}"
    )


print(
    "\nSHAP da classe positiva: OK"
)


# ============================================================
# 13. MAPEAR FEATURES SEM PARSING DE STRINGS
# ============================================================

def construir_mapa_features(
    preprocessador,
    numericas,
    categoricas
):

    nomes_transformados = np.asarray(
        preprocessador.get_feature_names_out(),
        dtype=object
    )


    registros = []


    # --------------------------------------------------------
    # Numéricas
    # --------------------------------------------------------

    for variavel in numericas:

        registros.append(
            {
                "Variavel_original":
                    variavel,

                "Tipo":
                    "numerica",

                "Categoria":
                    None,
            }
        )


    # --------------------------------------------------------
    # Categorias aprendidas pelo próprio OneHotEncoder
    # --------------------------------------------------------

    pipeline_cat = (
        preprocessador
        .named_transformers_[
            "cat"
        ]
    )


    encoder = (
        pipeline_cat
        .named_steps[
            "onehot"
        ]
    )


    categorias_encoder = encoder.categories_


    if len(
        categorias_encoder
    ) != len(
        categoricas
    ):

        raise RuntimeError(
            "Número de grupos no encoder diferente "
            "do número de variáveis categóricas."
        )


    for variavel, categorias in zip(
        categoricas,
        categorias_encoder
    ):

        for categoria in categorias:

            registros.append(
                {
                    "Variavel_original":
                        variavel,

                    "Tipo":
                        "categorica",

                    "Categoria":
                        str(
                            categoria
                        ),
                }
            )


    mapa = pd.DataFrame(
        registros
    )


    if len(
        mapa
    ) != len(
        nomes_transformados
    ):

        raise RuntimeError(
            "Mapa e número de features transformadas "
            "possuem tamanhos diferentes.\n"
            f"Mapa: {len(mapa)}\n"
            f"Features: {len(nomes_transformados)}"
        )


    mapa[
        "Feature_transformada"
    ] = nomes_transformados


    return mapa


mapa_features = construir_mapa_features(

    prep_final,

    NUMERICAS_FINAL,

    CATEGORICAS_FINAL
)


if len(
    mapa_features
) != shap_pos.shape[1]:

    raise RuntimeError(
        "Mapa SHAP incompatível com shap_pos."
    )


print(
    "\nMapeamento SHAP: OK"
)


print(
    "Features:",
    len(
        mapa_features
    )
)


print(
    "Variáveis originais:",
    mapa_features[
        "Variavel_original"
    ].nunique()
)


# ------------------------------------------------------------
# Mostrar __MISSING__
# ------------------------------------------------------------

missing = mapa_features[
    mapa_features[
        "Categoria"
    ]
    .astype(
        "string"
    )
    ==
    "__MISSING__"
]


if len(
    missing
):

    print(
        "\nCategorias __MISSING__ corretamente associadas:"
    )

    display(
        missing[
            [
                "Feature_transformada",
                "Variavel_original",
                "Categoria",
            ]
        ]
    )


mapa_features.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "08_SHAP_mapeamento_features.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 14. IMPORTÂNCIA GLOBAL SHAP
# ============================================================

mapa_features[
    "Mean_abs_SHAP"
] = np.mean(
    np.abs(
        shap_pos
    ),
    axis=0
)


shap_global = (
    mapa_features
    .groupby(
        "Variavel_original",
        as_index=False
    )
    .agg(
        Mean_abs_SHAP=(
            "Mean_abs_SHAP",
            "sum"
        ),

        N_features_transformadas=(
            "Feature_transformada",
            "size"
        ),
    )
    .sort_values(
        "Mean_abs_SHAP",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


shap_global[
    "Participacao_pct"
] = (

    shap_global[
        "Mean_abs_SHAP"
    ]

    /

    shap_global[
        "Mean_abs_SHAP"
    ].sum()

    *

    100
)


shap_global[
    "Ranking"
] = np.arange(
    1,
    len(
        shap_global
    )
    +
    1
)


shap_global.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "09_SHAP_importancia_global.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


print(
    "\nIMPORTÂNCIA GLOBAL SHAP"
)

display(
    shap_global
)


# ============================================================
# 15. FIGURA GLOBAL SHAP
# ============================================================

grafico = (
    shap_global
    .sort_values(
        "Participacao_pct",
        ascending=True
    )
)


fig, ax = plt.subplots(
    figsize=(
        9,
        7
    )
)


ax.barh(
    grafico[
        "Variavel_original"
    ],
    grafico[
        "Participacao_pct"
    ]
)


ax.set_xlabel(
    "Participação na importância global SHAP (%)"
)


ax.set_ylabel(
    "Variável"
)


ax.set_title(
    "Importância global SHAP — Random Forest final"
)


ax.grid(
    axis="x",
    alpha=0.25
)


fig.tight_layout()


fig.savefig(

    os.path.join(
        PASTA_FIGURAS,
        "04_SHAP_importancia_global.png"
    ),

    dpi=200,

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 16. AGRUPAR SHAP POR VARIÁVEL ORIGINAL
# ============================================================

indices_por_variavel = {}


for variavel in TODAS_VARIAVEIS_FINAL:

    indices_por_variavel[
        variavel
    ] = np.where(

        mapa_features[
            "Variavel_original"
        ].to_numpy()

        ==

        variavel
    )[0]


shap_por_variavel = {}


for variavel, indices in (
    indices_por_variavel.items()
):

    if len(
        indices
    ) == 0:

        continue


    shap_por_variavel[
        variavel
    ] = (
        shap_pos[
            :,
            indices
        ]
        .sum(
            axis=1
        )
    )


# ============================================================
# 17. BOOTSTRAP
# ============================================================

rng_boot = np.random.default_rng(
    SEED
)


def bootstrap_media(
    valores,
    b=200
):

    valores = np.asarray(
        valores,
        dtype=float
    )


    valores = valores[
        np.isfinite(
            valores
        )
    ]


    n = len(
        valores
    )


    if n < 2:

        return (
            np.nan,
            np.nan
        )


    medias = np.empty(
        b,
        dtype=float
    )


    for i in range(
        b
    ):

        idx = rng_boot.integers(
            0,
            n,
            size=n
        )


        medias[
            i
        ] = valores[
            idx
        ].mean()


    return (

        float(
            np.quantile(
                medias,
                0.025
            )
        ),

        float(
            np.quantile(
                medias,
                0.975
            )
        ),
    )


# ============================================================
# 18. SHAP POR CATEGORIA
# ============================================================

resultados_categoricos = []


for variavel in CATEGORICAS_FINAL:

    shap_var = shap_por_variavel[
        variavel
    ]


    categorias = (
        amostra_shap[
            variavel
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .reset_index(
            drop=True
        )
    )


    temp = pd.DataFrame(
        {
            "Categoria":
                categorias,

            "SHAP":
                shap_var,
        }
    )


    for categoria, grupo in temp.groupby(
        "Categoria",
        dropna=False
    ):

        valores = grupo[
            "SHAP"
        ].to_numpy()


        ic_inf, ic_sup = bootstrap_media(
            valores,
            BOOTSTRAP_SHAP
        )


        resultados_categoricos.append(
            {
                "Variavel":
                    variavel,

                "Categoria":
                    categoria,

                "n":
                    int(
                        len(
                            valores
                        )
                    ),

                "SHAP_medio":
                    float(
                        np.mean(
                            valores
                        )
                    ),

                "SHAP_mediano":
                    float(
                        np.median(
                            valores
                        )
                    ),

                "IC95_inf":
                    ic_inf,

                "IC95_sup":
                    ic_sup,
            }
        )


shap_categorias = pd.DataFrame(
    resultados_categoricos
)


shap_categorias = (
    shap_categorias
    .sort_values(
        [
            "Variavel",
            "SHAP_medio",
        ],
        ascending=[
            True,
            False,
        ]
    )
)


shap_categorias.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "10_SHAP_categorias.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 19. SHAP DAS VARIÁVEIS NUMÉRICAS
# ============================================================

resultados_numericos = []


for variavel in NUMERICAS_FINAL:

    shap_var = shap_por_variavel[
        variavel
    ]


    valores_raw = pd.to_numeric(
        amostra_shap[
            variavel
        ],
        errors="coerce"
    )


    temp = pd.DataFrame(
        {
            "Valor":
                valores_raw,

            "SHAP":
                shap_var,
        }
    ).dropna()


    if len(
        temp
    ) < 10:

        continue


    try:

        temp[
            "Faixa"
        ] = pd.qcut(
            temp[
                "Valor"
            ],
            q=10,
            duplicates="drop"
        )


        resumo = (
            temp
            .groupby(
                "Faixa",
                observed=True
            )
            .agg(
                n=("SHAP", "size"),
                Valor_medio=("Valor", "mean"),
                Valor_min=("Valor", "min"),
                Valor_max=("Valor", "max"),
                SHAP_medio=("SHAP", "mean"),
            )
            .reset_index()
        )


        resumo[
            "Variavel"
        ] = variavel


        resultados_numericos.append(
            resumo
        )


    except Exception as erro:

        print(
            f"Não foi possível criar faixas para "
            f"{variavel}: {erro}"
        )


if resultados_numericos:

    shap_numericas = pd.concat(
        resultados_numericos,
        ignore_index=True
    )


    shap_numericas.to_csv(

        os.path.join(
            PASTA_TABELAS,
            "11_SHAP_numericas.csv"
        ),

        index=False,

        encoding="utf-8-sig"
    )


else:

    shap_numericas = pd.DataFrame()


# ============================================================
# 20. ESTABILIDADE DA IMPORTÂNCIA SHAP
# ============================================================

variaveis_shap = (
    shap_global[
        "Variavel_original"
    ]
    .tolist()
)


matriz_importancia = np.zeros(

    (
        shap_pos.shape[
            0
        ],
        len(
            variaveis_shap
        )
    ),

    dtype=np.float32
)


for j, variavel in enumerate(
    variaveis_shap
):

    idx = indices_por_variavel[
        variavel
    ]


    matriz_importancia[
        :,
        j
    ] = (

        np.abs(
            shap_pos[
                :,
                idx
            ]
        )

        .sum(
            axis=1
        )
    )


rng_estabilidade = np.random.default_rng(
    SEED
)


ranks_boot = np.empty(

    (
        BOOTSTRAP_SHAP,
        len(
            variaveis_shap
        )
    ),

    dtype=float
)


n_shap = shap_pos.shape[
    0
]


for b in range(
    BOOTSTRAP_SHAP
):

    idx = rng_estabilidade.integers(
        0,
        n_shap,
        size=n_shap
    )


    medias = (
        matriz_importancia[
            idx
        ]
        .mean(
            axis=0
        )
    )


    ranks_boot[
        b
    ] = (
        pd.Series(
            -medias
        )
        .rank(
            method="average"
        )
        .to_numpy()
    )


estabilidade = pd.DataFrame(
    {
        "Variavel":
            variaveis_shap,

        "Rank_medio":
            ranks_boot.mean(
                axis=0
            ),

        "Rank_dp":
            ranks_boot.std(
                axis=0
            ),

        "Rank_min":
            ranks_boot.min(
                axis=0
            ),

        "Rank_max":
            ranks_boot.max(
                axis=0
            ),
    }
)


estabilidade = (
    estabilidade
    .sort_values(
        "Rank_medio"
    )
)


estabilidade.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "12_SHAP_estabilidade_ranking.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 21. EXPLICAÇÕES LOCAIS
# ============================================================

prob_shap = (
    rf_final
    .predict_proba(
        X_shap
    )[
        :,
        1
    ]
)


y_shap = (
    pd.to_numeric(
        amostra_shap[
            TARGET
        ],
        errors="raise"
    )
    .to_numpy(
        dtype=np.int8
    )
)


pred_shap = (
    prob_shap
    >=
    THRESHOLD_FINAL
).astype(
    np.int8
)


tipos = np.full(
    len(
        y_shap
    ),
    "",
    dtype=object
)


tipos[
    (y_shap == 1)
    &
    (pred_shap == 1)
] = "VP"


tipos[
    (y_shap == 0)
    &
    (pred_shap == 0)
] = "VN"


tipos[
    (y_shap == 0)
    &
    (pred_shap == 1)
] = "FP"


tipos[
    (y_shap == 1)
    &
    (pred_shap == 0)
] = "FN"


explicacoes_locais = []


for tipo in [
    "VP",
    "VN",
    "FP",
    "FN"
]:

    candidatos = np.where(
        tipos
        ==
        tipo
    )[0]


    if len(
        candidatos
    ) == 0:

        continue


    indice = int(
        candidatos[
            0
        ]
    )


    for variavel in variaveis_shap:

        explicacoes_locais.append(
            {
                "Tipo":
                    tipo,

                "Indice_amostra":
                    indice,

                "Y":
                    int(
                        y_shap[
                            indice
                        ]
                    ),

                "Predicao":
                    int(
                        pred_shap[
                            indice
                        ]
                    ),

                "Escore":
                    float(
                        prob_shap[
                            indice
                        ]
                    ),

                "Variavel":
                    variavel,

                "SHAP":
                    float(
                        shap_por_variavel[
                            variavel
                        ][
                            indice
                        ]
                    ),
            }
        )


pd.DataFrame(
    explicacoes_locais
).to_csv(

    os.path.join(
        PASTA_TABELAS,
        "13_SHAP_explicacoes_locais.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


print(
    "\nArquivos SHAP 08–13 concluídos."
)


# ============================================================
# 22. FUNÇÕES PARA O RANKING MUNICIPAL
# ============================================================

def limpar_categoria(
    serie
):

    return (
        serie
        .astype(
            "string"
        )
        .str.strip()
        .fillna(
            "__MISSING__"
        )
    )


def normalizar_municipio(
    serie
):

    return (
        serie
        .astype(
            "string"
        )
        .str.strip()
        .str.extract(
            r"(\d{6})",
            expand=False
        )
        .fillna(
            "__MISSING__"
        )
    )


def localizar_arquivos_ano(
    ano
):

    arquivos = sorted(
        glob.glob(
            os.path.join(
                PASTA_BASE,
                "**",
                f"RAIS_MODELO_FINAL_V3_{ano}_*.parquet"
            ),
            recursive=True
        )
    )


    if not arquivos:

        raise FileNotFoundError(
            f"Não encontrei arquivos V3 para {ano}."
        )


    return arquivos


# ============================================================
# 23. ESTATÍSTICAS MUNICIPAIS
# ============================================================

def estatisticas_municipais_ano(
    ano
):

    partes = []


    arquivos = localizar_arquivos_ano(
        ano
    )


    print(
        f"{ano}: {len(arquivos)} arquivo(s)"
    )


    for arquivo in arquivos:

        pf = pq.ParquetFile(
            arquivo
        )


        for batch in pf.iter_batches(

            batch_size=BATCH_SIZE,

            columns=[
                "UF",
                COL_MUN,
                TARGET,
            ]
        ):

            df = batch.to_pandas()


            df[
                "UF"
            ] = limpar_categoria(
                df[
                    "UF"
                ]
            )


            df[
                COL_MUN
            ] = normalizar_municipio(
                df[
                    COL_MUN
                ]
            )


            df[
                TARGET
            ] = pd.to_numeric(
                df[
                    TARGET
                ],
                errors="raise"
            )


            resumo = (
                df
                .groupby(
                    [
                        "UF",
                        COL_MUN,
                    ],
                    as_index=False
                )
                .agg(
                    N=(TARGET, "size"),
                    S=(TARGET, "sum"),
                )
            )


            partes.append(
                resumo
            )


            del df

            gc.collect()


    return (
        pd.concat(
            partes,
            ignore_index=True
        )
        .groupby(
            [
                "UF",
                COL_MUN,
            ],
            as_index=False
        )
        .agg(
            N=("N", "sum"),
            S=("S", "sum"),
        )
    )


# ============================================================
# 24. CALCULAR 2020–2024
# ============================================================

print(
    "\nCalculando estatísticas anuais para rankings..."
)


ESTATISTICAS = {}


for ano in [
    2020,
    2021,
    2022,
    2023,
    2024,
]:

    ESTATISTICAS[
        ano
    ] = estatisticas_municipais_ano(
        ano
    )


# ============================================================
# 25. COMBINAR HISTÓRICO E SUAVIZAR
# ============================================================

def construir_ranking_historico(
    anos
):

    acumulado = pd.concat(
        [
            ESTATISTICAS[
                ano
            ]

            for ano in anos
        ],
        ignore_index=True
    )


    municipios = (
        acumulado
        .groupby(
            [
                "UF",
                COL_MUN,
            ],
            as_index=False
        )
        .agg(
            N=("N", "sum"),
            S=("S", "sum"),
        )
    )


    uf = (
        municipios
        .groupby(
            "UF",
            as_index=False
        )
        .agg(
            N_UF=("N", "sum"),
            S_UF=("S", "sum"),
        )
    )


    uf[
        "Taxa_UF"
    ] = (
        uf[
            "S_UF"
        ]
        /
        uf[
            "N_UF"
        ]
    )


    municipios = municipios.merge(
        uf[
            [
                "UF",
                "Taxa_UF",
            ]
        ],
        on="UF",
        how="left"
    )


    municipios[
        "Taxa_bruta"
    ] = (
        municipios[
            "S"
        ]
        /
        municipios[
            "N"
        ]
    )


    municipios[
        COL_RISCO
    ] = (

        municipios[
            "S"
        ]

        +

        ALPHA_MUNICIPIO
        *
        municipios[
            "Taxa_UF"
        ]

    ) / (

        municipios[
            "N"
        ]

        +

        ALPHA_MUNICIPIO
    )


    municipios[
        COL_LOGN
    ] = np.log1p(
        municipios[
            "N"
        ]
    )


    municipios[
        "Taxa_bruta_pct"
    ] = (
        municipios[
            "Taxa_bruta"
        ]
        *
        100
    )


    municipios[
        "Risco_suavizado_pct"
    ] = (
        municipios[
            COL_RISCO
        ]
        *
        100
    )


    return municipios


# ============================================================
# 26. OBTER NOMES DOS MUNICÍPIOS DO IBGE
# ============================================================

def obter_nomes_ibge():

    try:

        import requests


        url = (
            "https://servicodados.ibge.gov.br/"
            "api/v1/localidades/municipios"
        )


        resposta = requests.get(
            url,
            timeout=60
        )


        resposta.raise_for_status()


        dados = resposta.json()


        linhas = []


        for item in dados:

            codigo_7 = str(
                item[
                    "id"
                ]
            )


            codigo_6 = codigo_7[
                :6
            ]


            try:

                uf = (
                    item[
                        "microrregiao"
                    ][
                        "mesorregiao"
                    ][
                        "UF"
                    ][
                        "sigla"
                    ]
                )

            except Exception:

                uf = None


            linhas.append(
                {
                    COL_MUN:
                        codigo_6,

                    "Municipio_nome":
                        item[
                            "nome"
                        ],

                    "UF_IBGE":
                        uf,
                }
            )


        tabela = pd.DataFrame(
            linhas
        )


        print(
            "\nNomes de municípios obtidos do IBGE:"
        )

        print(
            len(
                tabela
            )
        )


        return tabela


    except Exception as erro:

        print(
            "\nAVISO: não foi possível consultar o IBGE."
        )

        print(
            erro
        )

        print(
            "O ranking será salvo somente com código municipal."
        )


        return pd.DataFrame(
            columns=[
                COL_MUN,
                "Municipio_nome",
                "UF_IBGE",
            ]
        )


nomes_ibge = obter_nomes_ibge()


# ============================================================
# 27. PREPARAR RANKING
# ============================================================

def preparar_ranking(
    anos,
    periodo
):

    ranking = construir_ranking_historico(
        anos
    )


    if len(
        nomes_ibge
    ):

        ranking = ranking.merge(
            nomes_ibge,
            on=COL_MUN,
            how="left"
        )


    else:

        ranking[
            "Municipio_nome"
        ] = pd.NA


    ranking[
        "Periodo_historico"
    ] = periodo


    ranking[
        "Elegivel_ranking"
    ] = (
        ranking[
            "N"
        ]
        >=
        MIN_N_RANKING
    )


    elegiveis = (
        ranking[
            ranking[
                "Elegivel_ranking"
            ]
        ]
        .copy()
        .sort_values(
            COL_RISCO,
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    elegiveis[
        "Posicao_maior_risco"
    ] = np.arange(
        1,
        len(
            elegiveis
        )
        +
        1
    )


    top_maiores = elegiveis.head(
        TOP_N_RANKING
    ).copy()


    top_menores = (
        elegiveis
        .sort_values(
            COL_RISCO,
            ascending=True
        )
        .head(
            TOP_N_RANKING
        )
        .reset_index(
            drop=True
        )
    )


    top_menores[
        "Posicao_menor_risco"
    ] = np.arange(
        1,
        len(
            top_menores
        )
        +
        1
    )


    return (
        elegiveis,
        top_maiores,
        top_menores
    )


# ============================================================
# 28. RANKING 2020–2023
# ============================================================

(
    ranking_2020_2023,
    maiores_2020_2023,
    menores_2020_2023,

) = preparar_ranking(

    [
        2020,
        2021,
        2022,
        2023,
    ],

    "2020-2023"
)


ranking_2020_2023.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "14_ranking_municipal_2020_2023.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


maiores_2020_2023.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "15_top30_maiores_2020_2023.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


menores_2020_2023.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "16_top30_menores_2020_2023.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 29. RANKING 2020–2024
# ============================================================

(
    ranking_2020_2024,
    maiores_2020_2024,
    menores_2020_2024,

) = preparar_ranking(

    [
        2020,
        2021,
        2022,
        2023,
        2024,
    ],

    "2020-2024"
)


ranking_2020_2024.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "17_ranking_municipal_2020_2024.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


maiores_2020_2024.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "18_top30_maiores_2020_2024.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


menores_2020_2024.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "19_top30_menores_2020_2024.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


print(
    "\nTOP 30 MAIORES — 2020–2023"
)


display(
    maiores_2020_2023[
        [
            "Posicao_maior_risco",
            "Municipio_nome",
            "UF",
            COL_MUN,
            "N",
            "S",
            "Taxa_bruta_pct",
            "Risco_suavizado_pct",
        ]
    ]
)


print(
    "\nTOP 30 MENORES — 2020–2023"
)


display(
    menores_2020_2023[
        [
            "Posicao_menor_risco",
            "Municipio_nome",
            "UF",
            COL_MUN,
            "N",
            "S",
            "Taxa_bruta_pct",
            "Risco_suavizado_pct",
        ]
    ]
)


# ============================================================
# 30. DISTRIBUIÇÃO ANUAL
# ============================================================

distribuicao = []


for ano in [
    2020,
    2021,
    2022,
    2023,
    2024,
]:

    tabela = ESTATISTICAS[
        ano
    ]


    n = int(
        tabela[
            "N"
        ].sum()
    )


    s = int(
        tabela[
            "S"
        ].sum()
    )


    distribuicao.append(
        {
            "Ano":
                ano,

            "Vinculos":
                n,

            "Afastamentos":
                s,

            "Prevalencia":
                s / n,
        }
    )


# 2025: usamos o Y já salvo, sem reler a base inteira.
candidatos_y_2025 = glob.glob(
    os.path.join(
        PASTA_PREDICOES,
        "y_2025.npy"
    )
)


if candidatos_y_2025:

    y_2025 = np.load(
        candidatos_y_2025[
            0
        ],
        mmap_mode="r"
    )


    distribuicao.append(
        {
            "Ano":
                2025,

            "Vinculos":
                int(
                    len(
                        y_2025
                    )
                ),

            "Afastamentos":
                int(
                    np.sum(
                        y_2025
                    )
                ),

            "Prevalencia":
                float(
                    np.mean(
                        y_2025
                    )
                ),
        }
    )


distribuicao = pd.DataFrame(
    distribuicao
)


distribuicao.to_csv(

    os.path.join(
        PASTA_TABELAS,
        "20_distribuicao_anual.csv"
    ),

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 31. LER MÉTRICAS JÁ EXISTENTES
# ============================================================

metricas_2024 = pd.read_csv(
    ARQ_METRICAS_2024
)


mesma_sens = pd.read_csv(
    ARQ_MESMA_SENS
)


metricas_2025 = pd.read_csv(
    ARQ_METRICAS_2025
)


calibracao = pd.read_csv(
    ARQ_CALIBRACAO
)


# ============================================================
# 32. IDENTIFICAR LINHAS DOS MODELOS
# ============================================================

def linha_modelo(
    df,
    trecho
):

    mask = (
        df[
            "Modelo"
        ]
        .astype(
            str
        )
        .str.contains(
            trecho,
            case=False,
            regex=False
        )
    )


    encontrados = df[
        mask
    ]


    if len(
        encontrados
    ) == 0:

        raise RuntimeError(
            f"Modelo contendo '{trecho}' não encontrado."
        )


    return encontrados.iloc[
        0
    ]


m24_base = linha_modelo(
    metricas_2024,
    "original"
)


m24_final = linha_modelo(
    metricas_2024,
    "final"
)


m25_final = linha_modelo(
    metricas_2025,
    "final"
)


# Na tabela de mesma sensibilidade
m_same_base = linha_modelo(
    mesma_sens,
    "original"
)


m_same_final = linha_modelo(
    mesma_sens,
    "mesma"
)


# ============================================================
# 33. CALCULAR RESUMOS
# ============================================================

ganho_ap_abs = (
    m24_final[
        "Average_Precision"
    ]
    -
    m24_base[
        "Average_Precision"
    ]
)


ganho_ap_pct = (
    ganho_ap_abs
    /
    m24_base[
        "Average_Precision"
    ]
    *
    100
)


reducao_fp = (
    int(
        m_same_base[
            "FP"
        ]
    )
    -
    int(
        m_same_final[
            "FP"
        ]
    )
)


reducao_fp_pct = (
    reducao_fp
    /
    int(
        m_same_base[
            "FP"
        ]
    )
    *
    100
)


# ============================================================
# 34. LER CALIBRAÇÃO
# ============================================================

brier_linha = calibracao[
    calibracao[
        "Metrica"
    ]
    ==
    "Brier"
].iloc[
    0
]


log_linha = calibracao[
    calibracao[
        "Metrica"
    ]
    ==
    "Log_loss"
].iloc[
    0
]


# ============================================================
# 35. RESUMO PARA O TCC
# ============================================================

resumo = f"""
============================================================
MODELO FINAL — RESULTADOS CONSOLIDADOS
============================================================

ESPECIFICAÇÃO
------------------------------------------------------------

Random Forest original
+ risco histórico municipal suavizado
+ log do volume histórico municipal

Alpha da suavização:
{ALPHA_MUNICIPIO:.0f}


VALIDAÇÃO TEMPORAL — 2024
------------------------------------------------------------

RF ORIGINAL

AP:
{m24_base['Average_Precision']:.6f}

AP / prevalência:
{m24_base['AP_dividida_prevalencia']:.4f}

ROC-AUC:
{m24_base['ROC_AUC']:.6f}

Precisão:
{m24_base['Precision']:.4%}

Sensibilidade:
{m24_base['Recall']:.4%}

F1:
{m24_base['F1']:.6f}


RF FINAL

AP:
{m24_final['Average_Precision']:.6f}

AP / prevalência:
{m24_final['AP_dividida_prevalencia']:.4f}

ROC-AUC:
{m24_final['ROC_AUC']:.6f}

Precisão:
{m24_final['Precision']:.4%}

Sensibilidade:
{m24_final['Recall']:.4%}

F1:
{m24_final['F1']:.6f}


GANHO DO MODELO FINAL

Ganho absoluto da AP:
{ganho_ap_abs:.6f}

Ganho relativo da AP:
{ganho_ap_pct:.2f}%


COMPARAÇÃO NA MESMA SENSIBILIDADE
------------------------------------------------------------

Sensibilidade de referência:
{m_same_base['Recall']:.4%}

Precisão RF original:
{m_same_base['Precision']:.4%}

Precisão RF final:
{m_same_final['Precision']:.4%}

Falsos positivos RF original:
{int(m_same_base['FP']):,}

Falsos positivos RF final:
{int(m_same_final['FP']):,}

Redução absoluta de falsos positivos:
{reducao_fp:,}

Redução percentual de falsos positivos:
{reducao_fp_pct:.2f}%


AVALIAÇÃO TEMPORAL POSTERIOR — 2025
------------------------------------------------------------

AP:
{m25_final['Average_Precision']:.6f}

Prevalência:
{m25_final['Prevalencia']:.4%}

AP / prevalência:
{m25_final['AP_dividida_prevalencia']:.4f}

ROC-AUC:
{m25_final['ROC_AUC']:.6f}

Precisão:
{m25_final['Precision']:.4%}

Sensibilidade:
{m25_final['Recall']:.4%}

Especificidade:
{m25_final['Especificidade']:.4%}

F1:
{m25_final['F1']:.6f}

Acurácia balanceada:
{m25_final['Balanced_accuracy']:.6f}

Threshold congelado em 2024:
{THRESHOLD_FINAL:.8f}


CALIBRAÇÃO — 2025
------------------------------------------------------------

Brier RF final:
{brier_linha['RF_final']:.6f}

Brier preditor constante:
{brier_linha['Preditor_constante']:.6f}

Log-loss RF final:
{log_linha['RF_final']:.6f}

Log-loss preditor constante:
{log_linha['Preditor_constante']:.6f}


SHAP
------------------------------------------------------------

1ª variável:
{shap_global.iloc[0]['Variavel_original']}

Participação:
{shap_global.iloc[0]['Participacao_pct']:.2f}%

2ª variável:
{shap_global.iloc[1]['Variavel_original']}

Participação:
{shap_global.iloc[1]['Participacao_pct']:.2f}%

3ª variável:
{shap_global.iloc[2]['Variavel_original']}

Participação:
{shap_global.iloc[2]['Participacao_pct']:.2f}%


RANKING MUNICIPAL
------------------------------------------------------------

Ranking principal:
histórico 2020–2023

Alpha:
{ALPHA_MUNICIPIO:.0f}

Critério mínimo:
N >= {MIN_N_RANKING:,} vínculos históricos

Municípios elegíveis:
{len(ranking_2020_2023):,}

O risco municipal representa um indicador histórico suavizado
dos registros de afastamento por doença entre vínculos docentes
do município do estabelecimento.

Não deve ser interpretado como efeito causal nem como risco
clínico da população residente no município.
"""


ARQ_RESUMO = os.path.join(
    PASTA_RESULTADOS,
    "21_resumo_resultados_para_TCC.txt"
)


with open(
    ARQ_RESUMO,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        resumo
    )


print(
    resumo
)


# ============================================================
# 36. MANIFESTO FINAL
# ============================================================

manifesto_final = {

    "Base":
        "RAIS_BASE_MODELO_FINAL_V3",

    "Modelo":
        "Random Forest",

    "Treino":
        "2020-2023",

    "Validacao":
        2024,

    "Avaliacao_temporal_posterior":
        2025,

    "Threshold":
        THRESHOLD_FINAL,

    "Criterio_threshold":
        manifesto.get(
            "Criterio_threshold"
        ),

    "Alpha_municipal":
        ALPHA_MUNICIPIO,

    "Preditores_numericos":
        NUMERICAS_FINAL,

    "Preditores_categoricos":
        CATEGORICAS_FINAL,

    "Risco_municipal_temporal":
        True,

    "Municipio_direto":
        False,

    "Dummies_municipais":
        False,

    "Capital_Interior":
        False,

    "SHAP_amostra":
        int(
            len(
                amostra_shap
            )
        ),

    "SHAP_bootstrap":
        BOOTSTRAP_SHAP,

    "Ranking_min_N":
        MIN_N_RANKING,

    "Random_state":
        SEED,
}


ARQ_MANIFESTO_FINAL = os.path.join(
    PASTA_RESULTADOS,
    "22_manifesto_final.json"
)


with open(
    ARQ_MANIFESTO_FINAL,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifesto_final,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 37. AUDITORIA FINAL DOS ARQUIVOS
# ============================================================

esperados = [

    "08_SHAP_mapeamento_features.csv",
    "09_SHAP_importancia_global.csv",
    "10_SHAP_categorias.csv",
    "11_SHAP_numericas.csv",
    "12_SHAP_estabilidade_ranking.csv",
    "13_SHAP_explicacoes_locais.csv",
    "14_ranking_municipal_2020_2023.csv",
    "15_top30_maiores_2020_2023.csv",
    "16_top30_menores_2020_2023.csv",
    "17_ranking_municipal_2020_2024.csv",
    "18_top30_maiores_2020_2024.csv",
    "19_top30_menores_2020_2024.csv",
    "20_distribuicao_anual.csv",
]


print("\n" + "=" * 90)

print(
    "AUDITORIA FINAL"
)

print("=" * 90)


todos_ok = True


for nome in esperados:

    caminho = os.path.join(
        PASTA_TABELAS,
        nome
    )


    existe = os.path.exists(
        caminho
    )


    tamanho = (
        os.path.getsize(
            caminho
        )
        if existe
        else 0
    )


    print(
        f"{'OK' if existe else 'FALTA'} | "
        f"{nome} | "
        f"{tamanho:,} bytes"
    )


    if not existe:

        todos_ok = False


for caminho in [
    ARQ_RESUMO,
    ARQ_MANIFESTO_FINAL,
]:

    existe = os.path.exists(
        caminho
    )


    print(
        f"{'OK' if existe else 'FALTA'} | "
        f"{os.path.basename(caminho)}"
    )


    if not existe:

        todos_ok = False


print("\n" + "=" * 90)


if todos_ok:

    print(
        "COMPLEMENTO CONCLUÍDO COM SUCESSO."
    )

else:

    print(
        "ATENÇÃO: existem arquivos faltantes."
    )


print("=" * 90)


print(
    "\nPasta de tabelas:"
)

print(
    PASTA_TABELAS
)


print(
    "\nPasta principal:"
)

print(
    PASTA_RESULTADOS
)